# Error Analysis and Visualization

This notebook performs comprehensive error analysis and creates visualizations for model performance.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import cv2
from PIL import Image
import sys
import os

# Add src to path
sys.path.append('../src')

from models import get_model
from data_loader import get_data_loaders
from evaluate import MultiLabelEvaluator
from visualizations import (
    GradCAM, BoundingBoxVisualizer, PredictionVisualizer,
    create_error_analysis_plot
)

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

plt.rcParams['figure.figsize'] = (12, 8)

## Load Model and Data

In [ ]:
# Load data loaders
data_loaders = get_data_loaders(
    data_dir='../data/raw',
    metadata_file='../data/raw/metadata.csv',
    batch_size=32,
    num_workers=4,
    image_size=224
)

# Load trained model
model = get_model(
    model_type='multi_label',
    backbone='efficientnet_b0',
    num_diseases=7,
    num_pests=5,
    num_abiotic=5
)

checkpoint_path = '../experiments/checkpoints/best_model.pth'
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Model loaded from {checkpoint_path}")
else:
    print("No checkpoint found. Using untrained model.")

model = model.to(device)
model.eval()

## Collect Predictions for Analysis

In [ ]:
# Collect predictions and labels
all_predictions = []
all_labels = []
all_images = []
all_image_paths = []

with torch.no_grad():
    for images, labels in data_loaders['test']:
        images = images.to(device)
        outputs = model(images)
        
        if 'multi_label' in outputs:
            probs = torch.sigmoid(outputs['multi_label']).cpu().numpy()
            preds = (probs >= 0.5).astype(int)
            true = labels['multi_label'].numpy()
        elif 'diseases' in outputs:
            disease_probs = torch.sigmoid(outputs['diseases']).cpu().numpy()
            pest_probs = torch.sigmoid(outputs['pests']).cpu().numpy()
            abiotic_probs = torch.sigmoid(outputs['abiotic']).cpu().numpy()
            probs = np.concatenate([disease_probs, pest_probs, abiotic_probs], axis=1)
            preds = (probs >= 0.5).astype(int)
            true = labels['multi_label'].numpy()
        
        all_predictions.append(preds)
        all_labels.append(true)
        all_images.append(images.cpu())

all_predictions = np.vstack(all_predictions)
all_labels = np.vstack(all_labels)
all_images = torch.cat(all_images)

print(f"Collected {len(all_predictions)} predictions")

## Confusion Matrix Analysis

In [ ]:
# Label names
label_names = [
    'early_blight', 'late_blight', 'powdery_mildew', 'leaf_spot',
    'bacterial_spot', 'viral_infection', 'fungal_infection',
    'aphids', 'whiteflies', 'thrips', 'spider_mites', 'caterpillars',
    'nutrient_deficiency', 'water_stress', 'heat_stress',
    'salt_stress', 'light_stress'
]

# Create confusion matrices for each label
fig, axes = plt.subplots(3, 6, figsize=(20, 10))
axes = axes.flatten()

for i, label in enumerate(label_names):
    if i >= len(axes):
        break
    
    cm = confusion_matrix(all_labels[:, i], all_predictions[:, i])
    
    sns.heatmap(cm, annot=True, fmt='d', ax=axes[i], cmap='Blues', cbar=False)
    axes[i].set_title(label, fontsize=8)
    axes[i].set_xlabel('Predicted', fontsize=7)
    axes[i].set_ylabel('True', fontsize=7)
    axes[i].tick_params(labelsize=6)

# Hide unused subplots
for i in range(len(label_names), len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('../reports/figures/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## Error Analysis: False Positives and False Negatives

In [ ]:
# Analyze false positives and false negatives
def analyze_errors(predictions, labels, label_names):
    """Analyze false positives and false negatives per label"""
    error_analysis = {}
    
    for i, label in enumerate(label_names):
        true = labels[:, i]
        pred = predictions[:, i]
        
        # False positives: predicted 1, true 0
        fp = np.sum((pred == 1) & (true == 0))
        
        # False negatives: predicted 0, true 1
        fn = np.sum((pred == 0) & (true == 1))
        
        # True positives
        tp = np.sum((pred == 1) & (true == 1))
        
        # True negatives
        tn = np.sum((pred == 0) & (true == 0))
        
        error_analysis[label] = {
            'false_positives': fp,
            'false_negatives': fn,
            'true_positives': tp,
            'true_negatives': tn,
            'fp_rate': fp / (fp + tn) if (fp + tn) > 0 else 0,
            'fn_rate': fn / (fn + tp) if (fn + tp) > 0 else 0
        }
    
    return error_analysis

error_analysis = analyze_errors(all_predictions, all_labels, label_names)

# Print error analysis
print("Error Analysis:")
print("="*80)
for label, metrics in error_analysis.items():
    print(f"\n{label}:")
    print(f"  False Positives: {metrics['false_positives']} (Rate: {metrics['fp_rate']:.3f})")
    print(f"  False Negatives: {metrics['false_negatives']} (Rate: {metrics['fn_rate']:.3f})")

## Visualize Error Rates

In [ ]:
# Extract error rates
fp_rates = [error_analysis[label]['fp_rate'] for label in label_names]
fn_rates = [error_analysis[label]['fn_rate'] for label in label_names]

# Plot error rates
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(label_names))
width = 0.35

bars1 = ax.bar(x - width/2, fp_rates, width, label='False Positive Rate', color='#e74c3c')
bars2 = ax.bar(x + width/2, fn_rates, width, label='False Negative Rate', color='#3498db')

ax.set_xlabel('Labels')
ax.set_ylabel('Error Rate')
ax.set_title('False Positive and False Negative Rates per Label')
ax.set_xticks(x)
ax.set_xticklabels(label_names, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../reports/figures/error_rates.png', dpi=150, bbox_inches='tight')
plt.show()

## Identify Worst Performing Samples

In [ ]:
# Find samples with highest error
def calculate_sample_errors(predictions, labels):
    """Calculate error per sample"""
    # Hamming loss per sample
    errors = np.sum(predictions != labels, axis=1)
    return errors

sample_errors = calculate_sample_errors(all_predictions, all_labels)

# Get worst samples
worst_indices = np.argsort(sample_errors)[-10:][::-1]

print("Samples with highest errors:")
for idx in worst_indices:
    print(f"Sample {idx}: Error = {sample_errors[idx]}")
    print(f"  True labels: {np.where(all_labels])[0]}")
    print(f"  Predicted: {np.where(all_predictions[idx])[0]}")
    print()

## Grad-CAM Visualization

In [ ]:
# Initialize Grad-CAM
# Note: Need to specify the correct target layer for your model
# This is a demonstration - actual layer name depends on model architecture

# try:
#     grad_cam = GradCAM(model, target_layer='backbone.features.8')
#     
#     # Get a sample image
#     sample_image = all_images[0:1].to(device)
#     
#     # Generate CAM
#     cam = grad_cam.generate_cam(sample_image, target_class=0)
#     
#     # Denormalize image for visualization
#     mean = np.array([0.485, 0.456, 0.406])
#     std = np.array([0.229, 0.224, 0.225])
#     img_np = sample_image[0].cpu().numpy().transpose(1, 2, 0)
#     img_np = img_np * std + mean
#     img_np = np.clip(img_np, 0, 1)
#     
#     # Overlay CAM
#     overlay = grad_cam.overlay_cam((img_np * 255).astype(np.uint8), cam)
#     
#     # Display
#     fig, axes = plt.subplots(1, 3, figsize=(15, 5))
#     axes[0].imshow(img_np)
#     axes[0].set_title('Original Image')
#     axes[0].axis('off')
#     
#     axes[1].imshow(cam, cmap='jet')
#     axes[1].set_title('Grad-CAM Heatmap')
#     axes[1].axis('off')
#     
#     axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
#     axes[2].set_title('Overlay')
#     axes[2].axis('off')
#     
#     plt.tight_layout()
#     plt.show()
#     
# except Exception as e:
#     print(f"Grad-CAM error: {e}")
#     print("Grad-CAM requires proper target layer specification.")

## Prediction Confidence Analysis

In [ ]:
# Collect probability scores
all_probs = []

with torch.no_grad():
    for images, labels in data_loaders['test']:
        images = images.to(device)
        outputs = model(images)
        
        if 'multi_label' in outputs:
            probs = torch.sigmoid(outputs['multi_label']).cpu().numpy()
        elif 'diseases' in outputs:
            disease_probs = torch.sigmoid(outputs['diseases']).cpu().numpy()
            pest_probs = torch.sigmoid(outputs['pests']).cpu().numpy()
            abiotic_probs = torch.sigmoid(outputs['abiotic']).cpu().numpy()
            probs = np.concatenate([disease_probs, pest_probs, abiotic_probs], axis=1)
        
        all_probs.append(probs)

all_probs = np.vstack(all_probs)

# Analyze confidence distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall confidence distribution
axes[0].hist(all_probs.flatten(), bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Confidence Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Overall Confidence Distribution')
axes[0].axvline(0.5, color='red', linestyle='--', label='Threshold')
axes[0].legend()

# Confidence by correctness
correct_mask = (all_predictions == all_labels)
correct_probs = all_probs[correct_mask]
incorrect_probs = all_probs[~correct_mask]

axes[1].hist(correct_probs.flatten(), bins=50, alpha=0.5, label='Correct', color='#2ecc71')
axes[1].hist(incorrect_probs.flatten(), bins=50, alpha=0.5, label='Incorrect', color='#e74c3c')
axes[1].set_xlabel('Confidence Score')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Confidence by Prediction Correctness')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/confidence_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Performance Summary Dashboard

In [ ]:
# Create performance summary
evaluator = MultiLabelEvaluator(model, device=device)
metrics = evaluator.evaluate(data_loaders['test'])

# Display key metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Metric 1: F1 scores
f1_metrics = {k: v for k, v in metrics.items() if k.startswith('f1_') and '_' not in k.replace('f1_', '')}
if f1_metrics:
    axes[0, 0].barh(list(f1_metrics.keys()), list(f1_metrics.values()), color='#3498db')
    axes[0, 0].set_xlabel('F1 Score')
    axes[0, 0].set_title('Macro/Micro F1 Scores')
    axes[0, 0].set_xlim(0, 1)

# Metric 2: Health accuracy
health_acc = metrics.get('health_accuracy', 0)
axes[0, 1].bar(['Health Accuracy'], [health_acc], color='#2ecc71')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Health Status Classification')
axes[0, 1].set_ylim(0, 1)

# Metric 3: Severity metrics
severity_mae = metrics.get('severity_mae', 0)
severity_rmse = metrics.get('severity_rmse', 0)
axes[1, 0].bar(['MAE', 'RMSE'], [severity_mae, severity_rmse], color=['#f39c12', '#e74c3c'])
axes[1, 0].set_ylabel('Error')
axes[1, 0].set_title('Severity Estimation Error')

# Metric 4: Hamming loss
hamming = metrics.get('hamming_loss', 0)
axes[1, 1].bar(['Hamming Loss'], [hamming], color='#9b59b6')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].set_title('Multi-label Hamming Loss')
axes[1, 1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../reports/figures/performance_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
print("Error Analysis Summary:")
print("- Confusion matrices for each label")
print("- False positive/negative rate analysis")
print("- Worst performing sample identification")
print("- Grad-CAM visualization for model interpretability")
print("- Confidence distribution analysis")
print("- Performance dashboard with key metrics")
print("\nAll visualizations saved to reports/figures/")